# 第10课：July Weather-Signal Adaptive Hedge

本课建立项目的核心动态策略：3月先按预计产量的50%卖出December futures；7月观察PDSI和更新后的产量预测，再把目标套保比例调整为25%、50%或75%。

本课会分别记录3月初始头寸和7月调整头寸的交易价格与P&L，直接回应教授反馈。

## 0. 动态策略规则

3月所有情景都从50%开始。7月根据July PDSI分类：

| July PDSI状态 | 判断条件 | July目标比例 | 直观解释 |
|---|---|---:|---|
| Drought / dry | $PDSI<-0.2167$ | 25% | 预计产量较低，减少over-hedging风险 |
| Normal | $-0.2167\le PDSI\le2.31$ | 50% | 保持中性比例 |
| Wet | $PDSI>2.31$ | 75% | 预计产量较高，能够套保更多bushels |

两个threshold来自1996–2025历史July PDSI的terciles。规则是项目设定，不是Iowa State或CME发布的交易建议。

## 1. 两个时间点必须使用各自成交价格

3月建立的空头一直追踪到Harvest：

$$InitialPnL_i=N_{0,i}\times Q\times(F_{0,i}-F_{H,i})$$

7月新增或买回的合约使用July成交价：

$$JulyAdjustmentPnL_i=\Delta N_{July,i}\times Q\times(F_{July,i}-F_{H,i})$$

其中：

$$\Delta N_{July,i}=N_{TargetJuly,i}-N_{0,i}$$

- $ΔN>0$：7月继续卖出更多期货；
- $ΔN<0$：7月买回一部分原有空头；
- $ΔN=0$：不调整。

带符号的adjustment保证买回与追加卖出都使用同一个正确公式。

## 2. 导入工具并锁定参数

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

N_SIMULATIONS = 10_000
FARM_ACRES = 1_000
CONTRACT_SIZE_BUSHELS = 5_000
INITIAL_HEDGE_RATIO = 0.50

PDSI_LOW_THRESHOLD = -0.2166666666666667
PDSI_HIGH_THRESHOLD = 2.3099999999999996
DROUGHT_TARGET_RATIO = 0.25
NORMAL_TARGET_RATIO = 0.50
WET_TARGET_RATIO = 0.75

TRANSACTION_COST_PER_CONTRACT_SIDE = 25.00
INITIAL_MARGIN_PER_CONTRACT = 2_500.00
MARGIN_FINANCING_RATE_ANNUAL = 0.06
DAYS_PRESEASON_TO_JULY = 136
DAYS_JULY_TO_HARVEST = 108
MARGIN_LIQUIDITY_RESERVE = 50_000.00

LOWER_TAIL_PROBABILITY = 0.05

print('Initial hedge ratio:', INITIAL_HEDGE_RATIO)
print('PDSI thresholds:', PDSI_LOW_THRESHOLD, PDSI_HIGH_THRESHOLD)
print('July target ratios:', DROUGHT_TARGET_RATIO, NORMAL_TARGET_RATIO, WET_TARGET_RATIO)

## 3. 读取共同情景与Fixed Strategy结果

本课需要：

- 第7课共同情景，用于取得PDSI、July yield forecast和价格；
- 第9课fixed-strategy结果，用于与暂时首选Fixed 75%公平比较。

In [ ]:
lesson7_candidates = [
    Path.cwd() / 'lesson_07_outputs' / 'unhedged_baseline_10000.csv',
    Path.cwd().parent / 'lesson_07_outputs' / 'unhedged_baseline_10000.csv',
]
lesson9_candidates = [
    Path.cwd() / 'lesson_09_outputs' / 'fixed_strategy_scenario_results_50000.csv',
    Path.cwd().parent / 'lesson_09_outputs' / 'fixed_strategy_scenario_results_50000.csv',
]

lesson7_path = next((p for p in lesson7_candidates if p.exists()), None)
lesson9_path = next((p for p in lesson9_candidates if p.exists()), None)

if lesson7_path is None:
    raise FileNotFoundError('没有找到第7课共同情景CSV。请先运行Lesson_07。')
if lesson9_path is None:
    raise FileNotFoundError('没有找到第9课固定策略CSV。请先运行Lesson_09。')

scenarios = pd.read_csv(lesson7_path)
fixed_results = pd.read_csv(lesson9_path)
fixed75_results = (
    fixed_results[fixed_results['strategy'] == 'Fixed 75%']
    .sort_values('scenario_id')
    .reset_index(drop=True)
)

print('共同情景:', lesson7_path, len(scenarios))
print('Fixed 75%比较情景:', len(fixed75_results))

## 4. 检查输入

In [ ]:
required_columns = [
    'scenario_id',
    'july_pdsi',
    'preseason_expected_yield_bu_per_acre',
    'july_yield_forecast_bu_per_acre',
    'actual_production_bushels',
    'preseason_futures_usd_per_bushel',
    'july_futures_usd_per_bushel',
    'harvest_futures_usd_per_bushel',
    'cash_revenue_usd',
    'production_cost_usd',
]

missing = [c for c in required_columns if c not in scenarios.columns]
assert not missing, f'共同情景缺少列: {missing}'
assert len(scenarios) == N_SIMULATIONS
assert len(fixed75_results) == N_SIMULATIONS
assert np.array_equal(
    scenarios['scenario_id'].to_numpy(),
    fixed75_results['scenario_id'].to_numpy(),
)
assert scenarios[required_columns].isna().sum().sum() == 0

print('输入检查通过。')

## 5. 建立合约取整函数

In [ ]:
def rounded_contracts(quantity_bushels, contract_size=CONTRACT_SIZE_BUSHELS):
    quantity_bushels = np.asarray(quantity_bushels, dtype=float)
    return np.floor(np.maximum(quantity_bushels, 0.0) / contract_size + 0.5).astype(int)

## 6. 计算3月Initial Contracts

3月仍然使用preseason expected yield：

$$N_0=Round\left(\frac{50\%\times ExpectedYield\times Acres}{5{,}000}\right)$$

In [ ]:
expected_production = (
    scenarios['preseason_expected_yield_bu_per_acre'].to_numpy()
    * FARM_ACRES
)
initial_contracts = rounded_contracts(INITIAL_HEDGE_RATIO * expected_production)

print('Initial contracts unique values:', np.unique(initial_contracts))
print(f'Initial hedged bushels = {initial_contracts[0] * CONTRACT_SIZE_BUSHELS:,}')

## 7. 用July PDSI决定目标比例

In [ ]:
pdsi = scenarios['july_pdsi'].to_numpy()

july_target_ratio = np.select(
    [
        pdsi < PDSI_LOW_THRESHOLD,
        pdsi > PDSI_HIGH_THRESHOLD,
    ],
    [
        DROUGHT_TARGET_RATIO,
        WET_TARGET_RATIO,
    ],
    default=NORMAL_TARGET_RATIO,
)

weather_regime = np.select(
    [
        pdsi < PDSI_LOW_THRESHOLD,
        pdsi > PDSI_HIGH_THRESHOLD,
    ],
    [
        'Drought / Dry',
        'Wet',
    ],
    default='Normal',
)

ratio_counts = pd.Series(july_target_ratio).value_counts().sort_index()
print('July target-ratio counts:')
print(ratio_counts.to_string())

## 8. 使用July更新产量计算Target Contracts

注意这里不再使用3月expected yield，而是使用July时已经更新的产量预测：

$$N_{TargetJuly,i}=Round\left(\frac{TargetRatio_i\times JulyYieldForecast_i\times Acres}{5{,}000}\right)$$

$$\Delta N_i=N_{TargetJuly,i}-N_{0,i}$$

In [ ]:
july_forecast_production = (
    scenarios['july_yield_forecast_bu_per_acre'].to_numpy()
    * FARM_ACRES
)

final_contracts = rounded_contracts(
    july_target_ratio * july_forecast_production
)
july_adjustment_contracts = final_contracts - initial_contracts

adjustment_action = np.select(
    [
        july_adjustment_contracts < 0,
        july_adjustment_contracts > 0,
    ],
    [
        'Buy back shorts',
        'Sell more futures',
    ],
    default='No change',
)

print('Final-contract counts:')
print(pd.Series(final_contracts).value_counts().sort_index().to_string())
print()
print('July-adjustment counts:')
print(pd.Series(july_adjustment_contracts).value_counts().sort_index().to_string())

## 9. 观察前10个动态决策

In [ ]:
decision_table = pd.DataFrame({
    'scenario_id': scenarios['scenario_id'],
    'july_pdsi': pdsi,
    'weather_regime': weather_regime,
    'july_yield_forecast': scenarios['july_yield_forecast_bu_per_acre'],
    'july_target_ratio': july_target_ratio,
    'initial_contracts': initial_contracts,
    'final_contracts': final_contracts,
    'july_adjustment': july_adjustment_contracts,
    'action': adjustment_action,
})

print(decision_table.head(10).round(4).to_string(index=False))

## 10. 分别计算Initial P&L和July Adjustment P&L

In [ ]:
f0 = scenarios['preseason_futures_usd_per_bushel'].to_numpy()
f_july = scenarios['july_futures_usd_per_bushel'].to_numpy()
f_harvest = scenarios['harvest_futures_usd_per_bushel'].to_numpy()

initial_futures_pnl = (
    initial_contracts
    * CONTRACT_SIZE_BUSHELS
    * (f0 - f_harvest)
)

july_adjustment_pnl = (
    july_adjustment_contracts
    * CONTRACT_SIZE_BUSHELS
    * (f_july - f_harvest)
)

total_futures_pnl = initial_futures_pnl + july_adjustment_pnl

pnl_table = pd.DataFrame({
    'scenario_id': scenarios['scenario_id'],
    'initial_contracts': initial_contracts,
    'july_adjustment': july_adjustment_contracts,
    'F0': f0,
    'F_July': f_july,
    'F_Harvest': f_harvest,
    'initial_pnl': initial_futures_pnl,
    'july_adjustment_pnl': july_adjustment_pnl,
    'total_futures_pnl': total_futures_pnl,
})

print(pnl_table.head(10).round(2).to_string(index=False))

## 11. 手算Wet情景：Scenario 1追加卖出

Scenario 1的PDSI高于2.31，July target是75%。目标合约为32份，所以7月再卖出11份：

$$JulyPnL=11\times5{,}000\times(F_J-F_H)$$

In [ ]:
wet_row = 0
manual_wet_initial_pnl = (
    initial_contracts[wet_row] * CONTRACT_SIZE_BUSHELS
    * (f0[wet_row] - f_harvest[wet_row])
)
manual_wet_july_pnl = (
    july_adjustment_contracts[wet_row] * CONTRACT_SIZE_BUSHELS
    * (f_july[wet_row] - f_harvest[wet_row])
)

print('Scenario:', scenarios.loc[wet_row, 'scenario_id'])
print('PDSI:', pdsi[wet_row], 'Target ratio:', july_target_ratio[wet_row])
print('Adjustment:', july_adjustment_contracts[wet_row], 'contracts')
print(f'Initial P&L = ${manual_wet_initial_pnl:,.2f}')
print(f'July adjustment P&L = ${manual_wet_july_pnl:,.2f}')

assert july_adjustment_contracts[wet_row] > 0
assert np.isclose(manual_wet_july_pnl, july_adjustment_pnl[wet_row])
print('Wet情景手算通过。')

## 12. 手算Drought情景：Scenario 2买回空头

Scenario 2的PDSI低于−0.2167，July target降至25%。目标合约为10份，所以买回11份原有空头：

$$JulyPnL=(-11)\times5{,}000\times(F_J-F_H)$$

负的adjustment代表买回，并不代表P&L一定为负。

In [ ]:
dry_row = 1
manual_dry_initial_pnl = (
    initial_contracts[dry_row] * CONTRACT_SIZE_BUSHELS
    * (f0[dry_row] - f_harvest[dry_row])
)
manual_dry_july_pnl = (
    july_adjustment_contracts[dry_row] * CONTRACT_SIZE_BUSHELS
    * (f_july[dry_row] - f_harvest[dry_row])
)

print('Scenario:', scenarios.loc[dry_row, 'scenario_id'])
print('PDSI:', pdsi[dry_row], 'Target ratio:', july_target_ratio[dry_row])
print('Adjustment:', july_adjustment_contracts[dry_row], 'contracts')
print(f'Initial P&L = ${manual_dry_initial_pnl:,.2f}')
print(f'July adjustment P&L = ${manual_dry_july_pnl:,.2f}')

assert july_adjustment_contracts[dry_row] < 0
assert np.isclose(manual_dry_july_pnl, july_adjustment_pnl[dry_row])
print('Drought情景手算通过。')

## 13. 计算实施成本

Transaction sides包括：3月开仓、7月调整和Harvest平仓。

$$Sides=|N_0|+|\Delta N|+|N_H|$$

Margin financing cost则使用3月前持有的21份和7月调整后的final contracts。

In [ ]:
transaction_sides = (
    np.abs(initial_contracts)
    + np.abs(july_adjustment_contracts)
    + np.abs(final_contracts)
)
transaction_cost = transaction_sides * TRANSACTION_COST_PER_CONTRACT_SIDE

margin_financing_cost = (
    INITIAL_MARGIN_PER_CONTRACT
    * MARGIN_FINANCING_RATE_ANNUAL
    * (
        np.abs(initial_contracts) * DAYS_PRESEASON_TO_JULY / 365.0
        + np.abs(final_contracts) * DAYS_JULY_TO_HARVEST / 365.0
    )
)

print(f'Average transaction cost = ${transaction_cost.mean():,.2f}/farm')
print(f'Average margin financing cost = ${margin_financing_cost.mean():,.2f}/farm')

## 14. 计算Adaptive Profit

In [ ]:
cash_revenue = scenarios['cash_revenue_usd'].to_numpy()
production_cost = scenarios['production_cost_usd'].to_numpy()

gross_revenue_after_hedge = (
    cash_revenue
    + total_futures_pnl
    - transaction_cost
    - margin_financing_cost
)
profit = gross_revenue_after_hedge - production_cost
profit_per_acre = profit / FARM_ACRES

print(pd.DataFrame({
    'scenario_id': scenarios['scenario_id'],
    'cash_revenue': cash_revenue,
    'initial_pnl': initial_futures_pnl,
    'july_adjustment_pnl': july_adjustment_pnl,
    'transaction_cost': transaction_cost,
    'margin_cost': margin_financing_cost,
    'profit_per_acre': profit_per_acre,
}).head(10).round(2).to_string(index=False))

## 15. 计算Over-hedging与Margin-call Proxy

In [ ]:
final_hedged_bushels = final_contracts * CONTRACT_SIZE_BUSHELS
actual_production = scenarios['actual_production_bushels'].to_numpy()
overhedged = final_hedged_bushels > actual_production

july_mtm = initial_contracts * CONTRACT_SIZE_BUSHELS * (f0 - f_july)
harvest_segment_mtm = final_contracts * CONTRACT_SIZE_BUSHELS * (f_july - f_harvest)
margin_call_proxy = (
    ((-july_mtm) > MARGIN_LIQUIDITY_RESERVE)
    | ((-harvest_segment_mtm) > MARGIN_LIQUIDITY_RESERVE)
)

print(f'Probability overhedged = {overhedged.mean():.2%}')
print(f'Probability margin-call proxy = {margin_call_proxy.mean():.2%}')

## 16. 组合成标准Strategy Result表

In [ ]:
adaptive_results = pd.DataFrame({
    'scenario_id': scenarios['scenario_id'].to_numpy(),
    'strategy': 'Weather-signal 25/50/75%',
    'initial_hedge_ratio': INITIAL_HEDGE_RATIO,
    'july_target_hedge_ratio': july_target_ratio,
    'initial_contracts': initial_contracts,
    'july_adjustment_contracts': july_adjustment_contracts,
    'final_contracts': final_contracts,
    'actual_production_bushels': actual_production,
    'final_hedged_bushels': final_hedged_bushels,
    'cash_revenue_usd': cash_revenue,
    'initial_futures_pnl_usd': initial_futures_pnl,
    'july_adjustment_pnl_usd': july_adjustment_pnl,
    'total_futures_pnl_usd': total_futures_pnl,
    'transaction_cost_usd': transaction_cost,
    'margin_financing_cost_usd': margin_financing_cost,
    'gross_revenue_after_hedge_usd': gross_revenue_after_hedge,
    'profit_usd': profit,
    'profit_usd_per_acre': profit_per_acre,
    'overhedged': overhedged,
    'margin_call_proxy': margin_call_proxy,
})

print('Adaptive result rows:', len(adaptive_results))
print(adaptive_results.head(5).round(3).to_string(index=False))

## 17. 汇总Adaptive Strategy风险指标

In [ ]:
def risk_summary(series):
    series = pd.Series(series)
    p5 = series.quantile(LOWER_TAIL_PROBABILITY)
    return pd.Series({
        'Mean': series.mean(),
        'Std Dev': series.std(ddof=1),
        'Probability Below Zero': (series < 0).mean(),
        'P5': p5,
        'CVaR 5%': series[series <= p5].mean(),
        'Median': series.median(),
        'P95': series.quantile(0.95),
    })

adaptive_summary = risk_summary(adaptive_results['profit_usd_per_acre'])
print(adaptive_summary.round(4).to_string())
print(f'Average Initial Futures P&L = ${initial_futures_pnl.mean():,.2f}/farm')
print(f'Average July Adjustment P&L = ${july_adjustment_pnl.mean():,.2f}/farm')

## 18. 与Fixed 75%比较

Fixed 75%是第9课中合格fixed strategies的暂时首选。这里比较两者，但仍不做最终推荐。

In [ ]:
comparison = pd.DataFrame({
    'Fixed 75%': risk_summary(fixed75_results['profit_usd_per_acre']),
    'Weather-signal 25/50/75%': adaptive_summary,
}).T

comparison['Overhedge Probability'] = [
    fixed75_results['overhedged'].mean(),
    overhedged.mean(),
]
comparison['Margin-call Proxy'] = [
    fixed75_results['margin_call_proxy'].mean(),
    margin_call_proxy.mean(),
]

print(comparison.round(4).to_string())

## 19. 图形比较

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ratio_counts.plot(kind='bar', ax=axes[0], color=['#ED7D31', '#A5A5A5', '#70AD47'])
axes[0].set_title('July Target Hedge Ratio Counts')
axes[0].set_xlabel('Target hedge ratio')
axes[0].set_ylabel('Number of scenarios')
axes[0].tick_params(axis='x', rotation=0)

axes[1].hist(
    fixed75_results['profit_usd_per_acre'],
    bins=45,
    alpha=0.55,
    label='Fixed 75%',
    color='#4472C4',
)
axes[1].hist(
    profit_per_acre,
    bins=45,
    alpha=0.55,
    label='Weather-signal Adaptive',
    color='#70AD47',
)
axes[1].axvline(0, color='#C00000', linestyle='--')
axes[1].set_title('Profit Distribution Comparison')
axes[1].set_xlabel('USD per acre')
axes[1].set_ylabel('Number of scenarios')
axes[1].legend()

plt.tight_layout()
plt.show()

## 20. 必须通过的模型检查

In [ ]:
assert (initial_contracts == 21).all()
assert set(np.unique(july_target_ratio)) == {0.25, 0.50, 0.75}
assert set(np.unique(july_adjustment_contracts)) == {-11, 0, 11, 12}
assert (final_contracts == initial_contracts + july_adjustment_contracts).all()
assert np.allclose(
    initial_futures_pnl,
    initial_contracts * CONTRACT_SIZE_BUSHELS * (f0 - f_harvest),
)
assert np.allclose(
    july_adjustment_pnl,
    july_adjustment_contracts * CONTRACT_SIZE_BUSHELS * (f_july - f_harvest),
)
assert np.allclose(total_futures_pnl, initial_futures_pnl + july_adjustment_pnl)
assert len(adaptive_results) == N_SIMULATIONS
assert not adaptive_results.duplicated(['scenario_id', 'strategy']).any()
assert adaptive_results.isna().sum().sum() == 0

print('全部模型检查通过。')

## 21. 可重复性检查

In [ ]:
adaptive_p5 = pd.Series(profit_per_acre).quantile(0.05)
adaptive_cvar5 = pd.Series(profit_per_acre)[profit_per_acre <= adaptive_p5].mean()

expected = {
    'ratio_25_count': 3348,
    'ratio_50_count': 3318,
    'ratio_75_count': 3334,
    'mean_profit_per_acre': 13.131840405998949,
    'std_profit_per_acre': 87.43842499253981,
    'probability_below_zero': 0.4205,
    'p5_profit_per_acre': -132.07814743552314,
    'cvar5_profit_per_acre': -201.47493462655655,
    'mean_july_adjustment_pnl_farm': 1410.7815722277157,
    'probability_overhedged': 0.0,
    'probability_margin_call_proxy': 0.3463,
}

actual = {
    'ratio_25_count': int((july_target_ratio == 0.25).sum()),
    'ratio_50_count': int((july_target_ratio == 0.50).sum()),
    'ratio_75_count': int((july_target_ratio == 0.75).sum()),
    'mean_profit_per_acre': profit_per_acre.mean(),
    'std_profit_per_acre': profit_per_acre.std(ddof=1),
    'probability_below_zero': (profit_per_acre < 0).mean(),
    'p5_profit_per_acre': adaptive_p5,
    'cvar5_profit_per_acre': adaptive_cvar5,
    'mean_july_adjustment_pnl_farm': july_adjustment_pnl.mean(),
    'probability_overhedged': overhedged.mean(),
    'probability_margin_call_proxy': margin_call_proxy.mean(),
}

for key in expected:
    assert np.isclose(actual[key], expected[key], atol=1e-10), (key, actual[key], expected[key])

print('可重复性检查通过。')
print(pd.DataFrame({'Expected': expected, 'Actual': actual}).round(6).to_string())

## 22. 保存第10课结果

In [ ]:
OUTPUT_DIR = Path.cwd() / 'lesson_10_outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

results_path = OUTPUT_DIR / 'weather_signal_adaptive_10000.csv'
summary_path = OUTPUT_DIR / 'step_10_weather_signal_summary.json'

adaptive_results.to_csv(results_path, index=False)

summary_for_json = {
    'strategy': 'Weather-signal 25/50/75%',
    'initial_hedge_ratio': INITIAL_HEDGE_RATIO,
    'pdsi_thresholds': {
        'low': PDSI_LOW_THRESHOLD,
        'high': PDSI_HIGH_THRESHOLD,
    },
    'target_ratio_counts': {
        '0.25': int((july_target_ratio == 0.25).sum()),
        '0.50': int((july_target_ratio == 0.50).sum()),
        '0.75': int((july_target_ratio == 0.75).sum()),
    },
    'profit_summary_usd_per_acre': {
        k: float(v) for k, v in adaptive_summary.items()
    },
    'probability_overhedged': float(overhedged.mean()),
    'probability_margin_call_proxy': float(margin_call_proxy.mean()),
}

summary_path.write_text(json.dumps(summary_for_json, indent=2), encoding='utf-8')

print('已保存:', results_path)
print('已保存:', summary_path)

## 23. 本课结论与限制

Weather-signal adaptive strategy的平均利润约$13.13/acre，over-hedging probability为0%，margin-call proxy约34.63%。它确实根据July信息改变了头寸，并正确追踪两段交易价格。

但它的CVaR 5%约为−$201.47/acre，明显低于Fixed 75%的−$94.95/acre。因此在当前主模型下，它没有击败暂时首选Fixed 75%。

这不代表adaptive hedging没有价值。可能原因包括：

1. July价格回归解释力很弱；
2. Tercile rule是简单规则，不一定是最优threshold；
3. 在drought情景减少套保会保留更多价格风险；
4. Historical bootstrap样本只有30年。

下一步不能立刻结束：还需要加入“Updated 50%”策略，并完成July price beta = 0的robustness check。

**下一课：合并全部七种策略，应用同一选择标准，并进行关键robustness test。**